# SimCLR Analysis

Comprehensive analysis of a SimCLR model pretrained on STL-10 (96x96 images).

**Sections:**
- **A. Training Trajectories** -- loss, k-NN accuracy, and learning rate over epochs
- **B. Evaluation Results** -- k-NN sweep and linear probe (full + low-data)
- **C. Intermediate Layer Feature Maps** -- visualize ResNet-18 layer activations
- **D. SimCLR-Specific** -- augmented view pairs and t-SNE of learned features

In [ ]:
import sys
from pathlib import Path
import json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import transforms, datasets
from sklearn.manifold import TSNE

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from models.simclr import SimCLR, SimCLRAugmentation
from evaluation import extract_features
from evaluation.knn import knn_classify
from evaluation.linear_probe import LinearProbe
from utils.data import get_eval_loaders, STL10_MEAN, STL10_STD, DATA_DIR

sns.set_theme(style="whitegrid", font_scale=1.1)
device = "cuda" if torch.cuda.is_available() else "cpu"
CLASS_NAMES = ["airplane", "bird", "car", "cat", "deer", "dog", "horse", "monkey", "ship", "truck"]
RESULTS_DIR = PROJECT_ROOT / "results" / "simclr"

print(f"Device: {device}")
print(f"Results dir: {RESULTS_DIR}")

## A. Training Trajectories

Load the training history and plot loss, k-NN accuracy, and learning rate schedule over epochs.

In [ ]:
with open(RESULTS_DIR / "history.json") as f:
    history = json.load(f)

epochs = [h["epoch"] for h in history]
losses = [h["loss"] for h in history]
knn_accs = [h["knn_top1"] for h in history]
lrs = [h["lr"] for h in history]

print(f"Training epochs: {len(history)}")
print(f"Final loss: {losses[-1]:.4f}")
print(f"Best k-NN accuracy: {max(knn_accs):.4f} (epoch {epochs[np.argmax(knn_accs)]})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curve
axes[0].plot(epochs, losses, color="steelblue")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("SimCLR Training Loss (NT-Xent)")

# k-NN accuracy curve
axes[1].plot(epochs, [a * 100 for a in knn_accs], color="darkorange")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("k-NN Accuracy (%)")
axes[1].set_title("k-NN Top-1 Accuracy During Training")

# Learning rate schedule
axes[2].plot(epochs, lrs, color="seagreen")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Learning Rate")
axes[2].set_title("Learning Rate Schedule")

plt.tight_layout()
plt.show()

## B. Evaluation Results

Load the trained SimCLR model, extract features using the encoder (before projection head), and evaluate with k-NN and linear probe.

In [ ]:
# Load trained model
ckpt = torch.load(RESULTS_DIR / "best.pt", map_location=device, weights_only=True)
ckpt_args = ckpt["args"]
model = SimCLR(
    backbone=ckpt_args.get("backbone", "resnet18"),
    proj_output_dim=ckpt_args.get("proj_dim", 128),
    temperature=ckpt_args.get("temperature", 0.5),
)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device).eval()

print(f"Loaded SimCLR checkpoint")
print(f"  Backbone: {ckpt_args.get('backbone', 'resnet18')}")
print(f"  Latent dim: {model.latent_dim}")
print(f"  Proj dim: {ckpt_args.get('proj_dim', 128)}")
print(f"  Temperature: {ckpt_args.get('temperature', 0.5)}")

In [ ]:
# Load evaluation data (standard normalized images for SimCLR)
loaders = get_eval_loaders(batch_size=256)

# Extract features once using model.encoder (before projection head)
train_features, train_labels = extract_features(model.encoder, loaders["train"], device)
test_features, test_labels = extract_features(model.encoder, loaders["test"], device)

print(f"Train features: {tuple(train_features.shape)}")
print(f"Test features:  {tuple(test_features.shape)}")

### k-NN Accuracy Sweep

Evaluate k-NN classification across a range of k values using L2-normalized features.

In [ ]:
# L2-normalize features for cosine-similarity k-NN
train_feats_norm = F.normalize(train_features, dim=1)
test_feats_norm = F.normalize(test_features, dim=1)

k_values = [1, 5, 10, 20, 50, 100, 200]
knn_results = {}

for k in k_values:
    acc = knn_classify(train_feats_norm, train_labels, test_feats_norm, test_labels, k=k)
    knn_results[k] = acc
    print(f"k={k:>3d}  accuracy: {acc*100:.2f}%")

print(f"\nBest k-NN accuracy: {max(knn_results.values())*100:.2f}% at k={max(knn_results, key=knn_results.get)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(knn_results.keys()), [v * 100 for v in knn_results.values()],
        "o-", color="steelblue", linewidth=2, markersize=6)
ax.set_xscale("log")
ax.set_xlabel("k")
ax.set_ylabel("Accuracy (%)")
ax.set_title("SimCLR k-NN Accuracy vs k")
ax.set_xticks(k_values)
ax.set_xticklabels(k_values)
plt.tight_layout()
plt.show()

### Linear Probe -- Full Data and Low Data

Train a linear classifier on frozen SimCLR encoder features. Evaluate both on the full labeled training set and a low-data (1%) subset.

In [ ]:
# Full-data linear probe
probe_full = LinearProbe(feature_dim=model.latent_dim, num_classes=10, lr=0.1, epochs=100, device=device)
probe_full_losses = probe_full.fit(model.encoder, loaders["train"])
probe_full_acc = probe_full.evaluate(model.encoder, loaders["test"])
print(f"Linear probe (full data) test accuracy: {probe_full_acc*100:.2f}%")

# Low-data linear probe
probe_low = LinearProbe(feature_dim=model.latent_dim, num_classes=10, lr=0.1, epochs=100, device=device)
probe_low_losses = probe_low.fit(model.encoder, loaders["train_lowdata"])
probe_low_acc = probe_low.evaluate(model.encoder, loaders["test"])
print(f"Linear probe (low data) test accuracy:  {probe_low_acc*100:.2f}%")

print(f"\nLow-data training set size: {len(loaders['train_lowdata'].dataset)}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(range(1, len(probe_full_losses) + 1), probe_full_losses, color="steelblue", label="Full data")
ax1.plot(range(1, len(probe_low_losses) + 1), probe_low_losses, color="darkorange", label="Low data (1%)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Training Loss")
ax1.set_title("Linear Probe Training Loss")
ax1.legend()

# Summary bar chart
ax2.bar(["Full data", "Low data (1%)"], [probe_full_acc * 100, probe_low_acc * 100],
        color=["steelblue", "darkorange"], width=0.5)
ax2.set_ylabel("Test Accuracy (%)")
ax2.set_title("Linear Probe Test Accuracy")
ax2.set_ylim(0, 100)
for i, acc in enumerate([probe_full_acc, probe_low_acc]):
    ax2.text(i, acc * 100 + 1.5, f"{acc*100:.1f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

## C. Intermediate Layer Feature Maps

Visualize activations from each ResNet-18 layer (layer1 through layer4) for a single test image. Each layer shows the first 16 channels as a 4x4 grid.

In [ ]:
# Register forward hooks on each ResNet-18 layer
activations = {}

def make_hook(name):
    def hook(module, input, output):
        activations[name] = output.detach().cpu()
    return hook

hooks = []
for name in ["layer1", "layer2", "layer3", "layer4"]:
    h = getattr(model.encoder, name).register_forward_hook(make_hook(name))
    hooks.append(h)

# Get a single test image
test_ds = loaders["test"].dataset
test_img, test_label = test_ds[0]

# Forward pass to collect activations
with torch.no_grad():
    model.encoder(test_img.unsqueeze(0).to(device))

# Remove hooks
for h in hooks:
    h.remove()

# Print activation shapes
layer_names = ["layer1", "layer2", "layer3", "layer4"]
for name in layer_names:
    print(f"{name}: {tuple(activations[name].shape)}")

In [ ]:
def denormalize(tensor, mean=STL10_MEAN, std=STL10_STD):
    t = tensor.clone()
    for i, (m, s) in enumerate(zip(mean, std)):
        t[i].mul_(s).add_(m)
    return t.clamp(0, 1)

orig_img = denormalize(test_img).permute(1, 2, 0).numpy()
layer_channels = [64, 128, 256, 512]

fig, axes = plt.subplots(4, 5, figsize=(18, 14))

for row, (name, n_ch) in enumerate(zip(layer_names, layer_channels)):
    act = activations[name][0]  # (C, H, W)

    # Column 0: original image
    axes[row, 0].imshow(orig_img)
    axes[row, 0].set_title(f"Original ({CLASS_NAMES[test_label]})" if row == 0 else "")
    axes[row, 0].set_ylabel(f"{name}\n({n_ch} ch)", fontsize=11)
    axes[row, 0].set_xticks([])
    axes[row, 0].set_yticks([])

    # Columns 1-4: 16 channels arranged as four 2x2 grids
    for col in range(4):
        ch_start = col * 4
        grid_rows = []
        for r in range(2):
            grid_cols = []
            for c in range(2):
                ch = ch_start + r * 2 + c
                grid_cols.append(act[ch].numpy())
            grid_rows.append(np.concatenate(grid_cols, axis=1))
        grid = np.concatenate(grid_rows, axis=0)
        axes[row, col + 1].imshow(grid, cmap="viridis")
        axes[row, col + 1].set_title(f"ch {ch_start}-{ch_start+3}" if row == 0 else "")
        axes[row, col + 1].axis("off")

plt.suptitle("Intermediate Layer Feature Maps (first 16 channels per layer)", fontsize=14)
plt.tight_layout()
plt.show()

## D. SimCLR-Specific -- Augmented Views + t-SNE

### Augmented View Pairs

Show 4 images from the test set alongside their two augmented views produced by `SimCLRAugmentation()`.

In [ ]:
# Load raw PIL images for display and augmentation
raw_test_ds = datasets.STL10(root=str(DATA_DIR), split="test", download=False)
aug = SimCLRAugmentation()

fig, axes = plt.subplots(4, 3, figsize=(10, 13))
axes[0, 0].set_title("Original", fontsize=12)
axes[0, 1].set_title("View 1", fontsize=12)
axes[0, 2].set_title("View 2", fontsize=12)

for i in range(4):
    pil_img, label = raw_test_ds[i]
    view1, view2 = aug(pil_img)

    # Display original (convert PIL to tensor for display)
    axes[i, 0].imshow(pil_img)
    axes[i, 0].set_ylabel(CLASS_NAMES[label], fontsize=11, rotation=0, labelpad=55, va="center")
    axes[i, 0].set_xticks([])
    axes[i, 0].set_yticks([])

    # Display augmented views (denormalize since SimCLRAugmentation applies normalization)
    axes[i, 1].imshow(denormalize(view1).permute(1, 2, 0).numpy())
    axes[i, 1].axis("off")

    axes[i, 2].imshow(denormalize(view2).permute(1, 2, 0).numpy())
    axes[i, 2].axis("off")

plt.suptitle("SimCLR Augmented View Pairs", fontsize=14)
plt.tight_layout()
plt.show()

### t-SNE of Learned Features

Visualize the 2D t-SNE embedding of SimCLR encoder features from the training set, colored by class label.

In [ ]:
# Use the already-extracted train features (first 2000 samples)
n_subset = 2000
subset_feats = train_features[:n_subset].numpy()
subset_labels = train_labels[:n_subset].numpy()

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
embeddings = tsne.fit_transform(subset_feats)

print(f"t-SNE embedding shape: {embeddings.shape}")
print(f"Samples: {n_subset}, Classes: {len(np.unique(subset_labels))}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
cmap = plt.cm.tab10

for class_idx in range(10):
    mask = subset_labels == class_idx
    ax.scatter(
        embeddings[mask, 0], embeddings[mask, 1],
        c=[cmap(class_idx)], label=CLASS_NAMES[class_idx],
        s=15, alpha=0.7,
    )

ax.legend(markerscale=2.5, fontsize=10, loc="best")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.set_title("t-SNE of SimCLR Learned Features (2000 train samples)")
plt.tight_layout()
plt.show()